# Evidencias individuales de Trayectoria

Construye `data/evidencias/trayectoria/evidencias_trayectoria.csv` con el esquema común
`evidencia_id | persona_id | tipo_id | texto | atributos`. Las fechas (`fecha_inicio`, `fecha_fin`) y el detalle de
cada periodo (`periodos`) van dentro de `atributos`.

El mapeo de columnas y las decisiones están en `README.md` (esta carpeta). Esta etapa no genera embeddings.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "02_evidencias").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from evidencias.trayectoria import construir_evidencias_trayectoria

pd.set_option("display.max_colwidth", 200)

In [ ]:
evidencias, reporte, validacion = construir_evidencias_trayectoria(guardar=True)
print(json.dumps(reporte, ensure_ascii=False, indent=2))

## Validación del esquema (todas las filas deben tener `n_fallas = 0`)

In [ ]:
validacion

## Resumen por subtipo

In [ ]:
attrs = evidencias["atributos"].map(json.loads)
resumen = evidencias.assign(
    sin_fecha_fin=attrs.map(lambda a: a.get("fecha_fin") is None),
    n_periodos=attrs.map(lambda a: a.get("n_periodos", 1)),
).groupby("tipo_id").agg(
    evidencias=("evidencia_id", "size"),
    personas=("persona_id", "nunique"),
    periodos=("n_periodos", "sum"),
    evidencias_con_varios_periodos=("n_periodos", lambda s: int((s > 1).sum())),
    sin_fecha_fin=("sin_fecha_fin", "sum"),
    textos_distintos=("texto", "nunique"),
)
resumen

## Ejemplos por subtipo (texto + atributos)

In [ ]:
for tipo_id, grupo in evidencias.groupby("tipo_id"):
    print("\n=====", tipo_id)
    for _, r in grupo.sample(3, random_state=7).iterrows():
        print(f"- {r.evidencia_id} | {r.texto}")
        print("  ", json.dumps(json.loads(r.atributos), ensure_ascii=False))

## Atributos expandidos (útil para revisar coberturas por subtipo)

In [ ]:
def atributos_expandidos(tipo_id: str) -> pd.DataFrame:
    sub = evidencias[evidencias["tipo_id"] == tipo_id]
    attrs = pd.json_normalize(sub["atributos"].map(json.loads).tolist())
    return pd.concat([sub[["evidencia_id", "persona_id"]].reset_index(drop=True), attrs], axis=1)

for tipo_id in sorted(evidencias["tipo_id"].unique()):
    df = atributos_expandidos(tipo_id)
    print(f"\n{tipo_id}: % nulos por atributo")
    print((df.isna().mean() * 100).round(1).to_string())

## Textos más repetidos (baja información semántica)

In [ ]:
evidencias["texto"].value_counts().head(15)

## Una persona completa (cambiar el ID para revisar otros casos)

In [ ]:
PERSONA = evidencias["persona_id"].iloc[0]
una = evidencias[evidencias["persona_id"] == PERSONA]
attrs = una["atributos"].map(json.loads)
una.assign(
    fecha_inicio=attrs.map(lambda a: a.get("fecha_inicio")),
    fecha_fin=attrs.map(lambda a: a.get("fecha_fin")),
    n_periodos=attrs.map(lambda a: a.get("n_periodos", 1)),
)[["tipo_id", "fecha_inicio", "fecha_fin", "n_periodos", "texto"]]